This is a markdown cell. It describes the purpose of this notebook and its code.

This notebook will transform a PDF scan of an English record into a Word doc. The OCR process is done using the OpenAI API (ChatGPT). I attempted using Tesseract & OCRmyPDF previously. The results were not suitable for use with Scripts 1-7 without siginificant alterations to the scripts. 

Last updated by Kuba Kowalski on 18/02/2026, 15:30.

## Applying OCR to raw scans

In [42]:
# Install for convenience
!pip install ocrmypdf pdfplumber python-docx pymupdf requests

!sudo apt-get install -y tesseract-ocr tesseract-ocr-lat


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached urllib3-2.5.0-py3-none-any.whl.metadata (6.5 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached urllib3-2.5.0-py3-none-any.whl (129 kB)



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
Sudo is disabled on this machine. To enable it, go to the ]8;;ms-settings:developers\Developer Settings page]8;;\ in the Settings app


In [2]:
# Import packages
import os
import subprocess
from pathlib import Path
import pdfplumber
from docx import Document
import fitz  # PyMuPDF

# ChatGPT attempt at OCR

In [1]:
!pip install -U openai python-docx pymupdf

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ------- -------------------------------- 0.2/1.1 MB 6.3 MB/s eta 0:00:01
   ----------------------------- ---------- 0.8/1.1 MB 10.2 MB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 11.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
    --------------------------------------- 0.4/19.2 MB 22.4 MB/s eta 0:00:01
    --------------------------------------- 0.4/19.2 MB 22.4 MB/s eta 0:00:01
   - -------------------------------------- 0.5/19.2 MB 3.7 MB/s eta 0:00:06
   -- ------------------------------------- 1.2/19.2 MB 6.2 MB/s eta 0:00:03
   ---- ----------------------------------- 1.9/19.2 MB 8.2 MB/s eta 0:00:03
   ----- ---------------------------------- 2.7/19.2 MB 9.6 MB/s eta 0:00:02
   ------- -------------------------------- 3.5/19.2 MB 10.6 MB/s eta 0:00:02
   -------- ------------------------------- 3.9/19.2 MB 10.5 MB/s eta 0:00:02
   --------

  You can safely remove it manually.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# OCR all PDFs in a folder 

import base64
from pathlib import Path
from datetime import datetime
import fitz
from docx import Document
from docx.shared import Pt
import re
from openai import OpenAI

# ----------------------------
# CONFIG
# ----------------------------
API_KEY_PATH = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\open-ai-key.txt")
INPUT_DIR = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Esher\Scans Esher\Records")
OUTPUT_DIR = Path(r"C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Esher\OpenAI OCR Esher")

MODEL = "gpt-4.1" # Cheapest option, still good enough. Any performance improvements are best done using the prompt
RENDER_SCALE = 2.5
MAX_OUTPUT_TOKENS = 6000
OVERWRITE = False   # set True to re-run existing files

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

API_KEY = API_KEY_PATH.read_text(encoding="utf-8").strip()
if not API_KEY:
    raise SystemExit("API key missing")

client = OpenAI(api_key=API_KEY)

tag_re = re.compile(r"^<(H|P)>(.*)</\1>\s*$")

# ----------------------------
# HELPERS
# ----------------------------
def pdf_page_to_base64_png(pdf_path, page_num):
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    pix = page.get_pixmap(matrix=fitz.Matrix(RENDER_SCALE, RENDER_SCALE), alpha=False)
    img = pix.tobytes("png")
    doc.close()
    return base64.b64encode(img).decode("utf-8")

def ocr_page(image_b64, page_num, total_pages):
    prompt = (
        "Transcribe the MAIN BODY text from this scanned page.\n\n"
        "EXCLUDE: page numbers, headers, footnotes.\n\n"
        "OUTPUT FORMAT:\n"
        "- Section headings → <H>...</H>\n"
        "- Other lines → <P>...</P>\n"
        "- Preserve spelling, punctuation, numbers exactly.\n"
        "- Output ONLY tagged lines.\n"
    )

    resp = client.responses.create(
        model=MODEL,
        input=[{
            "role": "user",
            "content": [
                {"type": "input_text", "text": prompt},
                {"type": "input_image", "image_url": f"data:image/png;base64,{image_b64}"}
            ]
        }],
        max_output_tokens=MAX_OUTPUT_TOKENS
    )

    print(f"    OCR Page {page_num+1}/{total_pages} ✓")
    return (resp.output_text or "").strip()

def write_docx(pdf_name, pages, out_file):
    doc = Document()
    doc.add_paragraph(f"{pdf_name} — OCR output ({datetime.now().strftime('%Y-%m-%d %H:%M')})")

    for i, page_text in enumerate(pages, 1):
        doc.add_paragraph(f"--- Page {i} ---")

        for line in page_text.splitlines():
            line = line.strip()
            if not line:
                continue

            m = tag_re.match(line)
            if m:
                tag, content = m.group(1), m.group(2).strip()
                p = doc.add_paragraph()
                run = p.add_run(content)
                if tag == "H":
                    run.bold = True
                    run.font.size = Pt(12)
            else:
                doc.add_paragraph(line)

        if i < len(pages):
            doc.add_page_break()

    doc.save(out_file)

def process_pdf(pdf_path):
    out_file = OUTPUT_DIR / f"{pdf_path.stem}_ocr.docx"

    if out_file.exists() and not OVERWRITE:
        print(f"SKIP (exists): {pdf_path.name}")
        return

    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()

    print(f"\nProcessing {pdf_path.name} ({total_pages} pages)")

    pages = []
    for i in range(total_pages):
        img = pdf_page_to_base64_png(pdf_path, i)
        pages.append(ocr_page(img, i, total_pages))

    write_docx(pdf_path.name, pages, out_file)
    print(f"Saved: {out_file}")

# ----------------------------
# RUN ALL
# ----------------------------
pdfs = sorted(INPUT_DIR.glob("*.pdf"))
if not pdfs:
    raise SystemExit(f"No PDFs found in {INPUT_DIR}")

print(f"Found {len(pdfs)} PDFs")

for pdf in pdfs:
    process_pdf(pdf)

print("\nDone.")


Found 30 PDFs
SKIP (exists): Esher_1235.pdf
SKIP (exists): Esher_1245.pdf

Processing Esher_1247.pdf (10 pages)
    OCR Page 1/10 ✓
    OCR Page 2/10 ✓
    OCR Page 3/10 ✓
    OCR Page 4/10 ✓
    OCR Page 5/10 ✓
    OCR Page 6/10 ✓
    OCR Page 7/10 ✓
    OCR Page 8/10 ✓
    OCR Page 9/10 ✓
    OCR Page 10/10 ✓
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Esher\OpenAI OCR Esher\Esher_1247_ocr.docx

Processing Esher_1248.pdf (9 pages)
    OCR Page 1/9 ✓
    OCR Page 2/9 ✓
    OCR Page 3/9 ✓
    OCR Page 4/9 ✓
    OCR Page 5/9 ✓
    OCR Page 6/9 ✓
    OCR Page 7/9 ✓
    OCR Page 8/9 ✓
    OCR Page 9/9 ✓
Saved: C:\Users\kubak\OneDrive - Wageningen University & Research\WUR\2024-2025\research_assistant_ox\workfolder\input\Esher\OpenAI OCR Esher\Esher_1248_ocr.docx

Processing Esher_1251.pdf (7 pages)
    OCR Page 1/7 ✓
    OCR Page 2/7 ✓
    OCR Page 3/7 ✓
    OCR Page 4/7 ✓
    OCR Page 5/7 ✓
    OCR Page 6/7 ✓
    